In [ ]:
import yaml
import argparse
import sys
import os
import importlib

# 假设你项目根目录是 notebooks 的上上级目录
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

print(PROJECT_ROOT)
print(sys.path)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(sys.path)

#这个记得要删掉，Jupyter中因为没有这个python 的执行，所以只能加入sys.argv 参数去模仿。
sys.argv = ['notebook.py', '--job', 'top-produce-etl', '--ven', 'dev']

from utils.logger import setup_logging
from utils.spark_helper import create_spark_session,create_glue_context,detect_environment
from transform import clean_data

importlib.reload(clean_data)

def load_config(venv:str):
    config_path = f'../../config/config_{venv}.yaml'
    with open(config_path) as f:
        return yaml.safe_load(f)
    

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--job', required=True, help="need job name to run")
    parser.add_argument('--ven', default="dev", help="environment : dev or prod")

    return parser.parse_args()

def main():
    #get arguments of running environment
    pars = parse_args()
    
    #initialize log
    logger = setup_logging()
    logger.info(f"Starting job {pars.job} in {pars.ven} environment")

    #get all arguments through config
    configs = load_config(pars.ven)

    #creating sparksession
    if detect_environment():
        glue_context, spark = create_glue_context()
    else:
        spark = create_spark_session(pars.job, is_local=True)

    if pars.job == 'top-produce-etl':
        logger.info(configs['input']['city_path'])

        clean_data.run(spark, configs)
    else:
        logger.error(f"Job name {pars.job} not found!")
        sys.exit(1)

    logger.info(f"Job {pars.job} completed")

if __name__ == '__main__':
    main()

In [ ]:
import yaml
import argparse
import sys
import os
import importlib

# 假设你项目根目录是 notebooks 的上上级目录
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

print(PROJECT_ROOT)
print(sys.path)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(sys.path)

#这个记得要删掉，Jupyter中因为没有这个python 的执行，所以只能加入sys.argv 参数去模仿。
sys.argv = ['notebook.py', '--job', 'top-produce-etl', '--ven', 'dev']

from utils.logger import setup_logging
from utils.spark_helper import create_spark_session,create_glue_context,detect_environment
from transform import clean_data

import sys
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
import yaml
from readers import read_from_s3

spark = create_spark_session(is_local=True)

city_df = read_from_s3.read_s3_csv(spark, 's3a://jiazhi110-flink-staging-bucket/top-produce/data/city_info.csv', header=False, inferSchema=True)

produce_df = read_from_s3.read_s3_csv(spark, 's3a://jiazhi110-flink-staging-bucket/top-produce/data/product_info.csv', header=False, inferSchema=True)

user_visit_action_df = read_from_s3.read_s3_parquet(spark, 's3a://jiazhi110-flink-staging-bucket/user_action/dt=2019-07-17/')

# user_visit_action_df.show(200)

city_columns = ["city_id", "city_name", "area_name"]

produce_columns = ["produce_id", "produce_name", "extend_info"]

city_df = city_df.toDF(*city_columns)

produce_df = produce_df.toDF(*produce_columns)

# city_df.show()

# produce_df.show()

# 在用户行为表中，根据click_product_id、order_product_ids，pay_product_ids找出用户行为信息，并取名为behavior字段。
user_visit_action_df = user_visit_action_df.withColumn('behavior',
                                F.when(F.col('click_product_id').isNotNull() & (F.col('click_product_id').cast("string") != '') 
                                & (F.col('click_product_id').cast("string") != 'null') & (F.col('click_product_id').cast("string") != '-1'), 'click')
                                .otherwise('other'))

# 过滤掉其他的数据，只保留商品点击数据。
user_visit_action_df = user_visit_action_df.filter(F.col("behavior").isin('click'))

user_visit_action_df = user_visit_action_df.drop("user_id", "session_id", "page_id",
                            "action_time_ms", "search_keyword", "click_category_id",
                            "order_category_ids", "order_product_ids", "pay_category_ids",
                            "pay_product_ids")

# user_visit_action_df_filter = user_visit_action_df.sample(fraction=0.001, seed=24)

# user_visit_action_df.show(20)

user_visit_action_df.printSchema()

city_df.createOrReplaceTempView("city")
produce_df.createOrReplaceTempView("produce")
user_visit_action_df.createOrReplaceTempView("user_activity")

user_behavior_wide = spark.sql("""
    select user_activity.click_product_id, city.city_name, city.area_name, produce.produce_name
    from user_activity
    left join city on user_activity.city_id = city.city_id
    left join produce on user_activity.click_product_id = produce.produce_id
""")

# user_behavior_wide.show()

user_behavior_wide.createOrReplaceTempView("user_behavior_wide")

user_city_product_count = spark.sql("""
    select count(click_product_id) click_nums, area_name, city_name, produce_name
    from user_behavior_wide
    group by area_name, city_name, produce_name
    order by click_nums desc
""")

user_city_product_count.show()

user_city_product_count.createOrReplaceTempView("user_city_product_count")

# mysql 用法，这个是仿造spark SQL做的。  spark SQL 和 MySQL 在 grammar 上面 compatibility 有一些不同。

# select *
# from (
# 	select *,
# 	row_number() over (partition by area_name order by total_clicks desc ) as rn
# 	from (
# 		select area_name, produce_name,
# 			sum(click_nums) as total_clicks,
# 			GROUP_CONCAT(CONCAT(city_name, city_ratio, '%') order by city_ratio desc separator ',') as city_remark
# 		from (
# 				select click_nums, area_name, city_name, produce_name,
# 					ROUND(click_nums * 100.0 / SUM(click_nums) over (partition by area_name, produce_name), 1) as city_ratio
# 				from product_area_city_clicks
# 			) as city_ratio
# 		group by area_name , produce_name
# 	) as city_percent_remark
# ) as ranked_produce
# where rn <= 3
# order by total_clicks desc

# product_area_city_ratio_percent.show()

# product_area_city_ratio_percent.createOrReplaceTempView("product_area_city_ratio_percent")

# concat_ws 里的 `ws``就是 “with separator”

# | 函数                                    | 类型    | 作用          |
# | ------------------------------------- | ----- | ----------- |
# | `sum() over(...)`                     | 窗口函数  | 分组求和，用于占比计算 |
# | `row_number() over(...)`              | 窗口函数  | 排名，按点击数排序   |
# | `count(*) over(...)`                  | 窗口函数  | 统计城市数量      |
# | `collect_list()`                      | 聚合函数  | 把城市信息收集成数组  |
# | `named_struct()`                      | 结构函数  | 把多个字段组合成结构体 |
# | `array_sort()`                        | 数组函数  | 给数组排序       |
# | `slice(array, start, length)`         | 数组函数  | 取数组片段       |
# | `transform(array, x -> x.s)`          | 数组函数  | 遍历数组，提取字段   |
# | `concat_ws()`                         | 字符串函数 | 拼接数组元素成字符串  |
# | `round()`                             | 数学函数  | 四舍五入        |
# | `concat()`                            | 字符串函数 | 字符串拼接       |
# | `CASE WHEN ... THEN ... ELSE ... END` | 条件函数  | 条件逻辑判断      |

# -- ⚠️ 关于 count(*) vs count(col) 的说明
# -- 1. count(*) ：统计所有行（包括 NULL），和 count(任意非 NULL 列) 通常结果相同
# -- 2. count(col)：只统计该列非 NULL 的行数（如果列没有 NULL，等价于 count(*)）
# -- 3. count(distinct col)：统计该列去重后的唯一值数量
# --
# -- 为什么这里用 count(*) 而不是 count(city_name) 或 count(produce_color)？
# -- 因为 city 和 product 是 N:N 关系，如果直接 count(city_name)，
# -- 实际还是在数行数，不能准确代表“有多少个城市”。
# -- 如果以后又加了一个维度（比如 produce_color），他们的纬度关系是N:N:M
# -- count(*) 依然表示行数，不会因为新增维度而改变逻辑。
# --
# -- 如果业务确实需要“有多少个不同城市/颜色”，应该显式用：
# --   count(distinct city_name) over (partition by area_name, produce_name)
# --   count(distinct produce_color) over (partition by area_name, produce_name)
# --
# -- 结论：当前逻辑我们需要的是“行数”，所以选择 count(*)。


product_area_city_ratio_percent = spark.sql("""
    -- Step 1: 计算城市占比明细
    with city_ratio as (
        select
            area_name,
            produce_name,
            city_name,
            click_nums,
            round(click_nums * 100.0 / sum(click_nums) over (partition by area_name, produce_name), 1) as ratio_percent,
            row_number() over (partition by area_name, produce_name order by click_nums desc, city_name ASC) as city_rn,
            count(*) over (partition by area_name, produce_name) as city_cnt
        from user_city_product_count
    ),
    -- Step 2: 每个产品生成城市占比字符串
    product_city_str as (
        select
            area_name,
            produce_name,
            sum(click_nums) as total_clicks,
            -- 把前2名收集为字符串（collect_list -> array，concat_ws 把 array->string）
            CONCAT_WS('，',
                -- COLLECT_LIST(CASE WHEN city_rn <= 2 THEN CONCAT(city_name, ratio_percent, '%') END)
                transform(
                    slice(
                        array_sort(
                            collect_list(
                                named_struct('city_rn', city_rn, 's', concat(city_name, ratio_percent, '%'))
                            )
                        ),
                        1, 2
                    ),
                    x -> x.s
                )
            ) AS top2_str,

            -- 直接求 第3名及以后 的百分比和，作为"其他"
            CONCAT('其他', CAST(ROUND(SUM(CASE WHEN city_rn > 2 THEN ratio_percent ELSE 0 END), 1) AS STRING), '%') AS other_str,

            MAX(city_cnt) AS city_cnt
        FROM city_ratio
        group by area_name, produce_name
    ),
    -- Step 3: 每个地区给产品排序，取前 3
    ranked_product as (
        select
            area_name,
            produce_name,
            total_clicks,
            CASE 
                WHEN city_cnt > 2 THEN CONCAT(top2_str, '，', other_str)
                ELSE top2_str
            END AS city_remark,
            row_number() over (partition by area_name order by total_clicks desc) as rn
        from product_city_str
    )
    -- Step 4: 最终只取每个地区 top 3 产品
    select
        area_name,
        produce_name,
        total_clicks,
        city_remark
    from ranked_product
    where rn <= 3
    order by area_name, rn
""")

# product_area_city_ratio_percent = spark.sql("""
# -- =====================================================================================================================
# -- 功能说明：
# -- 本 SQL 旨在统计各个区域下，点击量排名前三的产品，并详细展示每个产品的点击量主要来自哪些城市。
# -- 对于每个产品，会列出点击量最高的 Top 2 城市及其占比，并将其他城市合并为“其他”项统一展示。
# --
# -- 最终产出：每个区域点击量最高的前3个产品，以及它们各自的城市点击分布情况（Top2 + 其他）。
# --
# -- SQL 逻辑分为四个步骤 (CTE - 公用表表达式):
# -- 1. city_ratio:          在每个 (区域, 产品) 分组内，计算每个城市的点击量占比和排名。
# -- 2. product_city_str:    在每个 (区域, 产品) 分组内，聚合所有城市信息，生成 Top2 城市字符串和“其他”字符串。
# -- 3. ranked_product:      在每个 (区域) 分组内，对产品按总点击量进行排名。
# -- 4. Final SELECT:        从排名结果中，筛选出每个区域的 Top 3 产品。
# -- =====================================================================================================================


# -- 步骤 1: 计算城市占比明细 (city_ratio)
# -- 目标：为后续聚合做数据准备。计算出每个产品在所属区域内，各个城市的点击数、点击占比、城市排名等详细指标。
# WITH city_ratio AS (
#     SELECT
#         area_name,
#         produce_name,
#         city_name,
#         click_nums,

#         -- 窗口函数：计算当前城市的点击量占该产品在该区域总点击量的百分比。
#         -- a. `SUM(...) OVER (...)` 计算窗口内的总和。
#         -- b. `PARTITION BY area_name, produce_name` 定义了窗口范围：同一个区域下的同一个产品。
#         -- c. `* 100.0` 确保进行浮点数除法，以计算百分比。
#         ROUND(click_nums * 100.0 / SUM(click_nums) OVER (PARTITION BY area_name, produce_name), 1) AS ratio_percent,

#         -- 窗口函数：为每个产品下的城市进行排名。
#         -- a. `ROW_NUMBER()` 生成一个从1开始的连续排名。
#         -- b. `ORDER BY click_nums DESC, city_name ASC` 定义排名规则：优先按点击量降序，如果点击量相同，则按城市名称升序（保证排名唯一性）。
#         ROW_NUMBER() OVER (PARTITION BY area_name, produce_name ORDER BY click_nums DESC, city_name ASC) AS city_rn,

#         -- 窗口函数：计算每个产品在该区域内总共有多少个城市有点击记录。
#         -- 这个字段用于后续判断是否需要显示“其他”项。
#         COUNT(*) OVER (PARTITION BY area_name, produce_name) AS city_cnt
#     FROM
#         user_city_product_count
# ),

# -- 步骤 2: 聚合城市信息，生成 Top2 和 "其他" 字符串 (product_city_str)
# -- 目标：将 `city_ratio` 中每个 (区域, 产品) 的多行城市数据，聚合成一行，并生成格式化的字符串。
# product_city_str AS (
#     SELECT
#         area_name,
#         produce_name,
#         SUM(click_nums) AS total_clicks, -- 计算每个产品在区域内的总点击量

#         -- [核心技巧] 生成 Top 2 城市的聚合字符串，例如 "北京15.0%，上海12.5%"
#         -- 这是一个现代 Spark SQL 的高级用法，将排序、切片、转换等操作在一次聚合中完成，非常高效。
#         CONCAT_WS('，', -- 步骤 e: 使用逗号将最终的字符串数组拼接成一个单一的字符串。
#             transform( -- 步骤 d: `transform` 函数遍历数组中的每个元素（这里是结构体），并应用一个转换逻辑。
#                        -- `x -> x.s` 表示对于每个结构体 `x`，只提取其 `s` 字段的值。最终得到一个字符串数组。
#                 slice( -- 步骤 c: `slice` 函数截取数组的一部分。这里从第 1 个元素开始，截取 2 个元素，即 Top 2。
#                     array_sort( -- 步骤 b: `array_sort` 对结构体数组进行排序。默认按结构体的第一个字段排序，这里即按 `city_rn` 升序排序。
#                         collect_list( -- 步骤 a: `collect_list` 将分组内的所有城市信息聚合成一个数组。
#                             -- `named_struct` 创建一个包含三个字段的结构体:
#                             -- 1. 'city_rn': 用于后续排序的依据。
#                             -- 2. 's': 预先拼接好的最终展示字符串。
#                             -- 这种“将排序键和值一起打包”的模式是关键，避免了对字符串进行排序的错误。
#                             named_struct('city_rn', city_rn, 's', CONCAT(city_name, ratio_percent, '%'))
#                         )
#                     ),
#                 1, 2)
#             , x -> x.s)
#         ) AS top2_str,

#         -- 计算排名第3及以后的城市占比总和，并格式化为“其他xx%”的字符串。
#         -- a. `SUM(CASE WHEN ...)`: 条件聚合，只对 `city_rn > 2` 的行进行 `ratio_percent` 的求和。
#         -- b. `ROUND(..., 1)`: 对求和结果四舍五入保留一位小数。
#         -- c. `CAST(... AS STRING)`: 将数值转换为字符串，以便与'其他'和'%'拼接。
#         CONCAT('其他', CAST(ROUND(SUM(CASE WHEN city_rn > 2 THEN ratio_percent ELSE 0 END), 1) AS STRING), '%') AS other_str,

#         -- 从分组中提取出 `city_cnt`。因为分组内每行的 `city_cnt` 都相同，所以用 MAX, MIN, FIRST 都可以。
#         MAX(city_cnt) AS city_cnt
#     FROM
#         city_ratio
#     GROUP BY
#         area_name, produce_name
# ),

# -- 步骤 3: 最终拼接城市备注，并对产品进行排名 (ranked_product)
# -- 目标：根据城市数量决定最终的备注字符串，并为每个区域内的产品按点击量进行排名。
# ranked_product AS (
#     SELECT
#         area_name,
#         produce_name,
#         total_clicks,
        
#         -- 根据 `city_cnt` 判断是否需要拼接 `other_str`。
#         -- 如果一个产品总共只有1个或2个城市，那么就不存在“其他”，直接显示 `top2_str` 即可。
#         CASE
#             WHEN city_cnt > 2 THEN CONCAT(top2_str, '，', other_str)
#             ELSE top2_str
#         END AS city_remark,

#         -- 窗口函数：对每个区域内的产品按总点击量进行排名。
#         ROW_NUMBER() OVER (PARTITION BY area_name ORDER BY total_clicks DESC) AS rn
#     FROM
#         product_city_str
# )

# -- 步骤 4: 筛选并排序最终结果
# -- 目标：从已排名的数据中，提取出每个区域的 Top 3 产品，并按指定顺序展示。
# SELECT
#     area_name,
#     produce_name,
#     total_clicks,
#     city_remark
# FROM
#     ranked_product
# WHERE
#     rn <= 3 -- 只保留每个区域排名前3的产品
# ORDER BY
#     area_name, rn -- 按区域分组，区域内按产品排名升序排列
# """
# )

product_area_city_ratio_percent.show(100, truncate=False)

product_area_city_ratio_percent.createOrReplaceTempView("product_area_city_ratio_percent")


/mnt/e/Top-produce-ETL/src
['/usr/lib/python39.zip', '/tmp/spark-3cfa9c85-1f6f-467d-a0d8-0dd04b20b4b7/userFiles-e7a317a0-b81c-4898-a1df-483f2f38a80f', '/usr/lib/python3.9', '/usr/lib/python3.9/lib-dynload', '', '/home/ubuntu/my-venvs/Top-produce-etl/lib/python3.9/site-packages', '/mnt/e/Top-produce-ETL/src']


root
 |-- click_product_id: integer (nullable = true)
 |-- city_id: integer (nullable = true)
 |-- hr: integer (nullable = true)
 |-- behavior: string (nullable = false)



+----------+---------+---------+------------+
|click_nums|area_name|city_name|produce_name|
+----------+---------+---------+------------+
|        28|     华南|     深圳|     商品_11|
|        26|     东北|     沈阳|     商品_52|
|        24|     西北|     西安|      商品_5|
|        24|     华南|     福州|     商品_95|
|        24|     西南|     成都|      商品_5|
|        22|     华东|     杭州|     商品_65|
|        22|     华东|     青岛|     商品_61|
|        22|     华南|     广州|      商品_1|
|        22|     华东|     苏州|     商品_50|
|        22|     东北|     沈阳|     商品_71|
|        22|     西南|     重庆|     商品_11|
|        22|     华北|     保定|     商品_81|
|        22|     华北|     天津|     商品_52|
|        22|     西北|     银川|      商品_2|
|        22|     华北|     郑州|     商品_99|
|        20|     华东|     南京|      商品_8|
|        20|     华中|     武汉|     商品_54|
|        20|     华南|     福州|     商品_92|
|        20|     华北|     郑州|     商品_55|
|        20|     华北|   石家庄|     商品_46|
+----------+---------+---------+------------+
only showing top 

+---------+------------+------------+---------------------------------+
|area_name|produce_name|total_clicks|city_remark                      |
+---------+------------+------------+---------------------------------+
|东北     |商品_52     |48          |沈阳54.2%，大连25.0%，其他20.8%  |
|东北     |商品_88     |48          |大连41.7%，沈阳37.5%，其他20.8%  |
|东北     |商品_44     |40          |哈尔滨45.0%，大连40.0%，其他15.0%|
|华东     |商品_10     |88          |杭州20.5%，南京18.2%，其他61.3%  |
|华东     |商品_85     |86          |上海20.9%，青岛18.6%，其他60.6%  |
|华东     |商品_21     |82          |杭州22.0%，南京19.5%，其他58.7%  |
|华中     |商品_4      |34          |长沙58.8%，武汉41.2%             |
|华中     |商品_100    |28          |武汉50.0%，长沙50.0%             |
|华中     |商品_29     |28          |武汉64.3%，长沙35.7%             |
|华北     |商品_68     |66          |北京27.3%，保定24.2%，其他48.5%  |
|华北     |商品_93     |64          |北京28.1%，天津25.0%，其他47.0%  |
|华北     |商品_46     |62          |石家庄32.3%，天津19.4%，其他48.4%|
|华南     |商品_95     |60          |福州40.0%，厦门23.3%，其他36.7% 

In [1]:
import pandas as pd
from pyspark.sql import SparkSession,DataFrame

spark = SparkSession.builder.master("local[*]").appName("text").getOrCreate()

city_df = spark.read.csv("../../test_data/city_info.txt", sep=" ")
produce_df = spark.read.csv("../../test_data/product_info.txt", sep="\t")
user_visit_action_df = spark.read.csv("../../test_data/user_visit_action.txt", sep="\t")

city_df.show()
# produce_df.show()
# user_visit_action_df.show()



# city_df.write.mode("overwrite").csv("../../test_data/city_info.csv")
# produce_df.write.mode("overwrite").csv("../../test_data/product_info.csv")
# user_visit_action_df.write.mode("overwrite").csv("../../test_data/user_visit_action.csv")



your 131072x1 screen size is bogus. expect trouble


KeyboardInterrupt: 

In [1]:
import yaml
import argparse
import sys
import os
import importlib

# 假设你项目根目录是 notebooks 的上上级目录
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

print(PROJECT_ROOT)
print(sys.path)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(sys.path)

#这个记得要删掉，Jupyter中因为没有这个python 的执行，所以只能加入sys.argv 参数去模仿。
sys.argv = ['notebook.py', '--job', 'top-produce-etl', '--ven', 'dev']

from utils.logger import setup_logging
from utils.spark_helper import create_spark_session,create_glue_context,detect_environment

import sys
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
import yaml
from readers import read_from_s3

spark = create_spark_session()

# df = read_from_s3.read_s3_csv(spark, 's3a://jiazhi110-flink-staging-bucket/output/top_products/', header=False, inferSchema=True)
df = read_from_s3.read_s3_parquet(spark, 's3a://jiazhi110-flink-staging-bucket/output/top_products/')

df.show(100 , truncate=False)

/mnt/e/Top-produce-ETL/src
['/usr/lib/python39.zip', '/usr/lib/python3.9', '/usr/lib/python3.9/lib-dynload', '', '/home/ubuntu/my-venvs/Top-produce-etl/lib/python3.9/site-packages']
['/usr/lib/python39.zip', '/usr/lib/python3.9', '/usr/lib/python3.9/lib-dynload', '', '/home/ubuntu/my-venvs/Top-produce-etl/lib/python3.9/site-packages', '/mnt/e/Top-produce-ETL/src']


your 131072x1 screen size is bogus. expect trouble


25/09/01 15:39:32 WARN Utils: Your hostname, MM-202011180033 resolves to a loopback address: 127.0.1.1; using 172.23.41.42 instead (on interface eth0)
25/09/01 15:39:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/09/01 15:39:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/09/01 15:39:39 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+---------+------------+------------+---------------------------------+
|area_name|produce_name|total_clicks|city_remark                      |
+---------+------------+------------+---------------------------------+
|东北     |商品_41     |183         |大连36.1%，哈尔滨33.9%，其他30.1%|
|东北     |商品_91     |179         |哈尔滨36.3%，大连31.8%，其他31.8%|
|东北     |商品_93     |178         |哈尔滨38.8%，大连38.2%，其他23.0%|
|华东     |商品_86     |407         |上海16.2%，杭州16.0%，其他67.8%  |
|华东     |商品_47     |402         |济南15.9%，杭州15.4%，其他68.7%  |
|华东     |商品_75     |402         |上海17.9%，青岛15.9%，其他66.1%  |
|华中     |商品_4      |131         |长沙54.2%，武汉45.8%             |
|华中     |商品_62     |128         |武汉50.8%，长沙49.2%             |
|华中     |商品_29     |126         |武汉51.6%，长沙48.4%             |
|华北     |商品_42     |296         |保定25.3%，郑州24.3%，其他50.3%  |
|华北     |商品_99     |292         |郑州25.7%，北京24.3%，其他50.0%  |
|华北     |商品_52     |291         |天津21.6%，北京21.0%，其他57.4%  |
|华南     |商品_23     |252         |厦门28.2%，福州25.0%，其他46.8%  